# Параллельные вычисления (2)

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы:
* Макрушин С.В. Лекция "Параллельные вычисления"
* https://nalepae.github.io/pandarallel/
    * https://github.com/nalepae/pandarallel/blob/master/docs/examples_windows.ipynb
    * https://github.com/nalepae/pandarallel/blob/master/docs/examples_mac_linux.ipynb
* https://requests.readthedocs.io/en/latest/
* https://docs.python.org/3/library/pathlib.html
* https://realpython.com/python-pathlib/
* https://realpython.com/python-gil/
* https://docs.python.org/3/library/multiprocessing.html#multiprocessing.pool.ThreadPool

## Задачи для совместного разбора

1. Выведите на экран слова из файла words_alpha, в которых есть две или более буквы "e" подряд.

In [ ]:
!pip install pandarallel

In [ ]:
import pandas as pd

words = (
    pd.read_csv("words_alpha.txt", header=None)[0]
    .dropna()
    .sample(frac=25, replace=True)
)

In [ ]:
words.head()

370069          zuffolo
340446    undiscardable
197128         nasicorn
176456            lutao
255526    psychotogenic
Name: 0, dtype: object

In [ ]:
words.shape

(9252575,)

In [ ]:
from pandarallel import pandarallel

pandarallel.initialize()

INFO: Pandarallel will run on 4 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.

https://nalepae.github.io/pandarallel/troubleshooting/


In [ ]:
%%file f.py
import re
def f(s):
    return re.search(r"e{2,}", s) is not None

Overwriting f.py


In [ ]:
from f import f

In [ ]:
%%time
r = words[words.map(f)]

Wall time: 12.2 s


In [ ]:
%%time
r = words[words.parallel_map(f)]

Wall time: 8.89 s


2. Загрузите данные о комментариях с сайта jsonplaceholder.typicode.com

![](https://i.imgur.com/AwiN8y6.png)

In [ ]:
import requests
import json
from pathlib import Path

In [ ]:
url = "https://jsonplaceholder.typicode.com/comments?postId=1"
r = requests.get(url)
r

<Response [200]>

In [ ]:
# r.text
# r.content
x = r.json()

In [ ]:
root = Path("./posts")
root.mkdir(exist_ok=True) #создаем папку
f = root / "123.json"
f.name, f.stem, f.parent

('123.json', '123', WindowsPath('posts'))

In [ ]:
%%time
base_url = "https://jsonplaceholder.typicode.com/comments?postId={post_id}"
for i in range(1, 51):
    url = base_url.format(post_id=i)
    r = requests.get(url)
    with open(root / f"{i}.json", "w", encoding="utf8") as fp:
        json.dump(r.json(), fp)

Wall time: 19.1 s


In [ ]:
def process(i):
    url = base_url.format(post_id=i)
    r = requests.get(url)
    with open(root / f"{i}.json", "w", encoding="utf8") as fp:
        json.dump(r.json(), fp)

In [ ]:
from multiprocessing.pool import ThreadPool

In [ ]:
%%time
with ThreadPool(processes=10) as pool:
    pool.map(process, range(1, 51))

Wall time: 2.64 s


3. Получите множество уникальных почтовых доменов.

![](https://i.imgur.com/ceY6guU.png)

In [ ]:
def process(comments):
    domains = set()
    for comment in comments:
        email = comment["email"]
        domain = email.split("@")[-1]
        domains.add(domain)
    return domains

In [ ]:
f

WindowsPath('posts/123.json')

In [ ]:
data = []
for f in root.iterdir(): #возвращает содержимое каталога
    with open(f, "r", encoding="utf8") as fp:
        comments = json.load(fp)
        data.append(comments)
data = data * 20_000
len(data)

1000000

In [ ]:
%%time
domains = set()
for comments in data:
    r = process(comments)
    domains.update(r)
len(domains)

Wall time: 2.84 s


249

In [ ]:
data[0]

[{'postId': 1,
  'id': 1,
  'name': 'id labore ex et quam laborum',
  'email': 'Eliseo@gardner.biz',
  'body': 'laudantium enim quasi est quidem magnam voluptate ipsam eos\ntempora quo necessitatibus\ndolor quam autem quasi\nreiciendis et nam sapiente accusantium'},
 {'postId': 1,
  'id': 2,
  'name': 'quo vero reiciendis velit similique earum',
  'email': 'Jayne_Kuhic@sydney.com',
  'body': 'est natus enim nihil est dolore omnis voluptatem numquam\net omnis occaecati quod ullam at\nvoluptatem error expedita pariatur\nnihil sint nostrum voluptatem reiciendis et'},
 {'postId': 1,
  'id': 3,
  'name': 'odio adipisci rerum aut animi',
  'email': 'Nikita@garfield.biz',
  'body': 'quia molestiae reprehenderit quasi aspernatur\naut expedita occaecati aliquam eveniet laudantium\nomnis quibusdam delectus saepe quia accusamus maiores nam est\ncum et ducimus et vero voluptates excepturi deleniti ratione'},
 {'postId': 1,
  'id': 4,
  'name': 'alias odio sit',
  'email': 'Lew@alysha.tv',
  'body'

In [ ]:
%%time

domains = set()
with ThreadPool(processes=10) as pool:
    r = pool.map(process, data)
for d in r:
    domains.update(d)
len(domains)

Wall time: 7.02 s


249

## Лабораторная работа 6

__При решении данных задач не подразумевается использования циклов или генераторов Python в ходе работы с пакетами `numpy` и `pandas`, если в задании не сказано обратного. Решения задач, в которых для обработки массивов `numpy` или структур `pandas` используются явные циклы (без согласования с преподавателем), могут быть признаны некорректными и не засчитаны.__

<p class="task" id="1"></p>

1\. Напишите функцию `f`, которая принимает на вход тэг и проверяет, удовлетворяет ли тэг следующему шаблону: `[любое число]-[любое слово]-or-less`. Возьмите файл `tag_nsteps_10m.csv`, примените функцию `f` при помощи метода _серий_ `map` к столбцу `tags` и посчитайте количество тэгов, подходящих под этот шаблон. Выведите количество подходящих тегов на экран. Измерьте время выполнения кода.

In [ ]:
import re
def f(tag: str) -> bool:
    pattern = '^[0-9]+-[a-zA-Z]+-or-less$'
    return re.search(pattern,str(tag)) is not None

In [ ]:
f('30-minutes-or-less'), f('30-342-or-less')

(True, False)

In [ ]:
import pandas as pd
df = pd.read_csv('C:/Users/Анастасия/3 семестр/ТОБД_2/tag_nsteps_10m.csv')

In [ ]:
%%time
res = df[df['tags'].map(f)]
print('Кол-во тегов = ', len(res))

Кол-во тегов =  288503
Wall time: 20.3 s


<!-- TODO -->

<p class="task" id="2"></p>

2\. Напишите функцию `parallel_map`, которая принимает на вход серию `s` `pd.Series` и функцию одного аргумента `f` и поэлементно применяет эту функцию к серии, распараллелив вычисления при помощи пакета `multiprocessing`. Логика работы функции `parallel_map` должна включать следующие действия:
* разбиение исходной серии на $K$ частей, где $K$ - количество ядер вашего процессора;
* параллельное применение функции `f` к каждой части при помощи метода _серии_ `map` c использованием нескольких подпроцессов;
* объединение результатов работы подпроцессов в одну серию.

Возьмите файл `tag_nsteps_10m.csv`, примените функцию `f` при помощи `parallel_map` к столбцу `tags` и посчитайте количество тэгов, подходящих под этот шаблон. Выведите количество подходящих тегов на экран. Измерьте время выполнения кода.

In [ ]:
import multiprocessing
import pandas as pd

In [ ]:
K = multiprocessing.cpu_count(); K

8

In [ ]:
%%file map_f.py
def map_f(x,f):
    return x.map(f)
import re
def f(tag: str) -> bool:
    pattern = '^[0-9]+-[a-zA-Z]+-or-less$'
    return re.search(pattern,str(tag)) is not None

Overwriting map_f.py


In [ ]:
from map_f import map_f , f

In [ ]:
[f]*8

[<function map_f.f(tag: str) -> bool>,
 <function map_f.f(tag: str) -> bool>,
 <function map_f.f(tag: str) -> bool>,
 <function map_f.f(tag: str) -> bool>,
 <function map_f.f(tag: str) -> bool>,
 <function map_f.f(tag: str) -> bool>,
 <function map_f.f(tag: str) -> bool>,
 <function map_f.f(tag: str) -> bool>]

In [ ]:
import numpy as np
def parallel_map(s: pd.Series, f: callable) -> pd.Series:
    data_split = np.array_split(s,8)
    with multiprocessing.Pool(processes=6) as pool:
        r = pool.starmap(map_f, zip(data_split, [f]*8))
    return pd.concat(r)

In [ ]:
%%time
sum(parallel_map(df['tags'], f))

Wall time: 10.4 s


288503

<p class="task" id="3"></p>

3\. Используя пакет `pandarallel`, примените функцию `f` из задания 1 к столбцу `tags` таблицы, с которой вы работали в этом задании. Посчитайте количество тэгов, подходящих под описанный шаблон. Выведите на экран полученный результат. Измерьте время выполнения кода.

In [ ]:
from pandarallel import pandarallel

pandarallel.initialize()

INFO: Pandarallel will run on 4 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.

https://nalepae.github.io/pandarallel/troubleshooting/


In [ ]:
def f(tag: str) -> bool:
    import re
    pattern = '^[0-9]+-[a-zA-Z]+-or-less$'
    return re.search(pattern,str(tag)) is not None

In [ ]:
%%time
df['tags'].parallel_map(f).sum()

Wall time: 10.4 s


288503

<p class="task" id="4"></p>

4\. Сайт [DummyJSON](https://dummyjson.com/) позволяет получить набор данных о товарах в виде JSON. Воспользовавшись пакетом `requests`, получите данные о __50 товарах__ и создайте словарь, где ключом является название товара (title), а значением - список ссылок на изображения этого товара. При создании словаря замените символ `/` в названии на пробел.

Выведите на экран количество элементов полученного словаря.

In [ ]:
import requests
url = 'https://dummyjson.com/products?limit=50'
responce = requests.get(url)
dict_json = responce.json()
print(dict_json)
dict_new = {}

{'products': [{'id': 1, 'title': 'iPhone 9', 'description': 'An apple mobile which is nothing like apple', 'price': 549, 'discountPercentage': 12.96, 'rating': 4.69, 'stock': 94, 'brand': 'Apple', 'category': 'smartphones', 'thumbnail': 'https://i.dummyjson.com/data/products/1/thumbnail.jpg', 'images': ['https://i.dummyjson.com/data/products/1/1.jpg', 'https://i.dummyjson.com/data/products/1/2.jpg', 'https://i.dummyjson.com/data/products/1/3.jpg', 'https://i.dummyjson.com/data/products/1/4.jpg', 'https://i.dummyjson.com/data/products/1/thumbnail.jpg']}, {'id': 2, 'title': 'iPhone X', 'description': 'SIM-Free, Model A19211 6.5-inch Super Retina HD display with OLED technology A12 Bionic chip with ...', 'price': 899, 'discountPercentage': 17.94, 'rating': 4.44, 'stock': 34, 'brand': 'Apple', 'category': 'smartphones', 'thumbnail': 'https://i.dummyjson.com/data/products/2/thumbnail.jpg', 'images': ['https://i.dummyjson.com/data/products/2/1.jpg', 'https://i.dummyjson.com/data/products/2/2

In [ ]:
for i in dict_json['products']:
    titles = i['title'].replace('/', ' ')
    image_list = i['images']
    dict_new[titles] = image_list
print(len(dict_new))

50


In [ ]:
dict_new

{'iPhone 9': ['https://i.dummyjson.com/data/products/1/1.jpg',
  'https://i.dummyjson.com/data/products/1/2.jpg',
  'https://i.dummyjson.com/data/products/1/3.jpg',
  'https://i.dummyjson.com/data/products/1/4.jpg',
  'https://i.dummyjson.com/data/products/1/thumbnail.jpg'],
 'iPhone X': ['https://i.dummyjson.com/data/products/2/1.jpg',
  'https://i.dummyjson.com/data/products/2/2.jpg',
  'https://i.dummyjson.com/data/products/2/3.jpg',
  'https://i.dummyjson.com/data/products/2/thumbnail.jpg'],
 'Samsung Universe 9': ['https://i.dummyjson.com/data/products/3/1.jpg'],
 'OPPOF19': ['https://i.dummyjson.com/data/products/4/1.jpg',
  'https://i.dummyjson.com/data/products/4/2.jpg',
  'https://i.dummyjson.com/data/products/4/3.jpg',
  'https://i.dummyjson.com/data/products/4/4.jpg',
  'https://i.dummyjson.com/data/products/4/thumbnail.jpg'],
 'Huawei P30': ['https://i.dummyjson.com/data/products/5/1.jpg',
  'https://i.dummyjson.com/data/products/5/2.jpg',
  'https://i.dummyjson.com/data/pr

<p class="task" id="5"></p>

5\. Напишите функцию `download_product_imgs`, которая создает папку c названием товара внутри каталога `imgs` (сам каталог `imgs` может быть создан любым удобным способом до начала работы) и сохраняет в нее изображения. Название для файла изображения извлеките из URL.

Воспользовавшись этой функцией, скачайте изображения всех продуктов. Выведите на экран общее количество загруженных файлов. Для отслеживания хода выполнения кода используйте пакет `tqdm`.

In [ ]:
# пример кода для скачивания картинки
url = "https://png.pngtree.com/png-vector/20201229/ourmid/pngtree-a-british-short-blue-and-white-cat-png-image_2654518.jpg"
img = requests.get(url).content
with open("cat.jpg", "wb") as fp:
    fp.write(img)

In [ ]:
import os
os.mkdir('imgs')

In [ ]:
import tqdm
def download_product_imgs(title, imgs):
    '''
    title - название товара
    imgs - список ссылок на изображения товара
    '''
    os.mkdir(f'imgs/{title}')
    for url in tqdm.tqdm(imgs):
        img = requests.get(url).content
        img_title = url.split('/')[-1]
        with open(f"imgs/{title}/{img_title}", "wb") as f:
            f.write(img)

In [ ]:
for product in dict_new:
    download_product_imgs(product,dict_new[product])

100%|████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:02<00:00,  1.11it/s]


In [ ]:
counter = 0
for product in dict_new:
    counter += len(dict_new[product])
counter

223

In [ ]:
import os
counter = 0
for root, dirs, files in os.walk("C:/Users/Анастасия/3 семестр/ТОБД_2/imgs"):
    for name in files:
        counter+=1
counter

223

<p class="task" id="6"></p>

6\. Создайте функцию `download_product_imgs_processes` на основе функции `download_product_imgs`, добавив в нее вывод сообщения следующего вида: `Process ID: <ID текущего процесса>`. Для определения ID процесса воспользуйтесь функцией `multiprocessing.current_process()`.

Решите задачу 5, распараллелив вычисления при помощи процессов. Вместо корневого каталога `imgs` используйте `imgs_processes`. Выведите на экран общее количество загруженных файлов. Измерьте время выполнения кода.

In [ ]:
import os
os.mkdir('imgs_processes')

In [ ]:
import multiprocessing

In [ ]:
%%file task_download_product_imgs_processes.py
import os
import multiprocessing
import requests
def download_product_imgs_processes(title, imgs):
    '''
    title - название товара
    imgs - список ссылок на изображения товара
    '''
    os.mkdir(f'imgs_processes/{title}')
    for url in imgs:
        img = requests.get(url).content
        img_title = url.split('/')[-1]
        with open(f"imgs_processes/{title}/{img_title}", "wb") as f:
            f.write(img)

    return f"Process ID: {multiprocessing.current_process().name}"

Overwriting task_download_product_imgs_processes.py


In [ ]:
from task_download_product_imgs_processes import download_product_imgs_processes

In [ ]:
keyses = dict_new.keys()
values = dict_new.values()

In [ ]:
%%time
with multiprocessing.Pool(processes=8) as pool:
    r = pool.starmap(download_product_imgs_processes, zip(keyses, values))
r

Wall time: 28.2 s


['Process ID: SpawnPoolWorker-22',
 'Process ID: SpawnPoolWorker-22',
 'Process ID: SpawnPoolWorker-23',
 'Process ID: SpawnPoolWorker-23',
 'Process ID: SpawnPoolWorker-26',
 'Process ID: SpawnPoolWorker-26',
 'Process ID: SpawnPoolWorker-24',
 'Process ID: SpawnPoolWorker-24',
 'Process ID: SpawnPoolWorker-25',
 'Process ID: SpawnPoolWorker-25',
 'Process ID: SpawnPoolWorker-29',
 'Process ID: SpawnPoolWorker-29',
 'Process ID: SpawnPoolWorker-27',
 'Process ID: SpawnPoolWorker-27',
 'Process ID: SpawnPoolWorker-28',
 'Process ID: SpawnPoolWorker-28',
 'Process ID: SpawnPoolWorker-23',
 'Process ID: SpawnPoolWorker-23',
 'Process ID: SpawnPoolWorker-26',
 'Process ID: SpawnPoolWorker-26',
 'Process ID: SpawnPoolWorker-24',
 'Process ID: SpawnPoolWorker-24',
 'Process ID: SpawnPoolWorker-25',
 'Process ID: SpawnPoolWorker-25',
 'Process ID: SpawnPoolWorker-22',
 'Process ID: SpawnPoolWorker-22',
 'Process ID: SpawnPoolWorker-27',
 'Process ID: SpawnPoolWorker-27',
 'Process ID: SpawnP

In [ ]:
counter = 0
for root, dirs, files in os.walk("C:/Users/Анастасия/3 семестр/ТОБД_2/imgs_processes"):
    for name in files:
        counter+=1
counter

223

<p class="task" id="7"></p>

7\. Создайте функцию `download_product_imgs_threads` на основе функции `download_product_imgs`, добавив в нее вывод сообщения следующего вида: `Process ID: <ID текущего процесса> Thread ID: <ID текущего потока>`. Для определения ID потока воспользуйтесь функцией `threading.get_ident`.

Решите задачу 5, распараллелив вычисления при помощи потоков. Вместо корневого каталога `imgs` используйте `imgs_threads`. Выведите на экран общее количество загруженных файлов. Измерьте время выполнения кода.

In [ ]:
from multiprocessing.pool import ThreadPool
import threading

In [ ]:
os.mkdir('imgs_threads')

In [ ]:
%%file download_product_imgs_threads.py
import os
import multiprocessing
import requests
import threading
def download_product_imgs_threads(title, imgs):
    '''
    title - название товара
    imgs - список ссылок на изображения товара
    '''
    os.mkdir(f'imgs_threads/{title}')
    for url in imgs:
        img = requests.get(url).content
        img_title = url.split('/')[-1]
        with open(f"imgs_threads/{title}/{img_title}", "wb") as f:
            f.write(img)

    return f"Process ID:{multiprocessing.current_process().name} Thread ID: {str(threading.get_ident())}"

Overwriting download_product_imgs_threads.py


In [ ]:
from download_product_imgs_threads import download_product_imgs_threads

In [ ]:
%%time
with ThreadPool(processes=8) as pool:
    r = pool.starmap(download_product_imgs_threads, zip(keyses, values))
r

Wall time: 28.4 s


['Process ID:MainProcess Thread ID: 17196',
 'Process ID:MainProcess Thread ID: 17196',
 'Process ID:MainProcess Thread ID: 17788',
 'Process ID:MainProcess Thread ID: 17788',
 'Process ID:MainProcess Thread ID: 3384',
 'Process ID:MainProcess Thread ID: 3384',
 'Process ID:MainProcess Thread ID: 6820',
 'Process ID:MainProcess Thread ID: 6820',
 'Process ID:MainProcess Thread ID: 5968',
 'Process ID:MainProcess Thread ID: 5968',
 'Process ID:MainProcess Thread ID: 14996',
 'Process ID:MainProcess Thread ID: 14996',
 'Process ID:MainProcess Thread ID: 8848',
 'Process ID:MainProcess Thread ID: 8848',
 'Process ID:MainProcess Thread ID: 17824',
 'Process ID:MainProcess Thread ID: 17824',
 'Process ID:MainProcess Thread ID: 17788',
 'Process ID:MainProcess Thread ID: 17788',
 'Process ID:MainProcess Thread ID: 3384',
 'Process ID:MainProcess Thread ID: 3384',
 'Process ID:MainProcess Thread ID: 6820',
 'Process ID:MainProcess Thread ID: 6820',
 'Process ID:MainProcess Thread ID: 17196',


In [ ]:
counter = 0
for root, dirs, files in os.walk("C:/Users/Анастасия/3 семестр/ТОБД_2/imgs_threads"):
    for name in files:
        counter+=1
counter

223

<p class="task" id="8"></p>

8\. Напишите функцию `create_2d_list`, которая создает матрицу размера `m` на `n` (__в виде списка списков__) вещественных чисел из стандартного нормального распределения. Напишите функцию `sum_by_chunk`, которая принимает на вход несколько строк этой матрицы (тоже в виде списка списков) и находит сумму элементов.

Используя данную функцию, решите задачу поиска суммы по всей матрице тремя способами:
* передав в функцию `sum_by_chunk` всю матрицу целиком;
* распараллелив вычисления при помощи процессов по следующему принципу: матрица разбивается на части (например, по 1тыс. строк); процессы независимо друг от друга обрабатывают эти части; после завершения работы всех процессов результаты агрегируются для получения результата для всей матрицы;
* распараллелив вычисления при помощи потоков аналогичным способом.

Для демонстрации результата создайте матрицу достаточно большого размера (не менее 100 тыс. строк), выведите на экран результаты работы трех вариантов решения и измерьте время выполнения каждого из них.

В данном задании разрешается использовать пакет `numpy` только для создания матрицы. В этом случае необходимо преобразовать ее к списку списков до начала работы.

In [ ]:
from random import randint
import multiprocessing
from multiprocessing.pool import ThreadPool
import numpy as np

def create_2d_list(m,n):
    lst=[[np.random.normal() for i in range(n)] for i in range(m)]
    return lst
Matrix1 = create_2d_list(100_000, 6)

In [ ]:
%%file sum_by_chunk.py
def sum_by_chunk(Matrix):
    summa_all = 0
    for elem in Matrix:
        sum_stroka = sum(elem)
        summa_all += sum_stroka
    return summa_all

Overwriting sum_by_chunk.py


In [ ]:
from sum_by_chunk import sum_by_chunk

In [ ]:
%%time
sum_by_chunk(Matrix1)

Wall time: 44 ms


1527.2573341976458

In [ ]:
ListOfMatrix = []
for i in range(100):
    ListOfMatrix.append(Matrix1[i*1000:(i+1)*1000])
len(ListOfMatrix)

100

In [ ]:
%%time
with multiprocessing.Pool(processes=8) as pool:
    r = pool.map(sum_by_chunk, ListOfMatrix)
sum(r)

Wall time: 573 ms


1527.2573341976133

In [ ]:
%%time
with ThreadPool(processes = 8) as pool:
    r = pool.map(sum_by_chunk, ListOfMatrix)
sum(r)

Wall time: 59.4 ms


1527.2573341976133

<p class="task" id="9"></p>

9\. Напишите функцию `create_2d_arr`, которая создает матрицу размера `m` на `n` (__в виде массива numpy__) вещественных чисел из стандартного нормального распределения. Напишите функцию `sum_by_chunk_np`, которая принимает на вход несколько строк этой матрицы (тоже в виде массива) и находит сумму элементов.

Используя данную функцию, решите задачу поиска суммы по всей матрице тремя способами аналогично задаче 8.

Для демонстрации результата создайте матрицу достаточно большого размера (не менее 100 тыс. строк), выведите на экран результаты работы трех вариантов решения и измерьте время выполнения каждого из них.

В данном задании при поиска суммы не используйте встроенную функцию `sum`, вместо этого используйте возможности `numpy`.

In [ ]:
import random
import numpy as np
def create_2d_arr(n,m):
    Matrix = np.random.randn(n, m)
    return Matrix
Matrix2 = create_2d_arr(100_000,6)

In [ ]:
def sum_by_chunk_np(Matrix):
    res = np.sum(Matrix)
    return res

In [ ]:
ArrayOfMatrix = np.asarray(np.array_split(Matrix2, 100))

In [ ]:
%%time
sum_by_chunk_np(Matrix2)

Wall time: 1.99 ms


-175.92347320940235

In [ ]:
%%file sum_by_chunk_np.py
import numpy as np
def sum_by_chunk_np(Matrix):
    res = np.sum(Matrix)
    return res

Overwriting sum_by_chunk_np.py


In [ ]:
from sum_by_chunk_np import sum_by_chunk_np

In [ ]:
%%time
with multiprocessing.Pool(processes=8) as pool:
    r = pool.map(sum_by_chunk_np, ArrayOfMatrix)
sum(r)

Wall time: 1.02 s


-175.9234732094023

In [ ]:
%%time
with ThreadPool(processes = 8) as pool:
    r = pool.map(sum_by_chunk_np, ArrayOfMatrix)
sum(r)

Wall time: 20 ms


-175.9234732094023